# LAD-GENERIC validation

Run selected benchmark tasks against adapters stored in Google Drive.
For `open_ended`, the runner evaluates 30 fixed questions, records perplexity and repetition, and can use GPT-5 to blind-rank all models' responses to each prompt.

In [ ]:
%cd /content
!if [ -d lad-generic/.git ]; then git -C lad-generic pull; else git clone https://github.com/RuurdKuiper/lad-generic.git; fi
%cd /content/lad-generic
!python -m pip install --upgrade pip
!python -m pip install --no-cache-dir '.[cuda,evaluation]'
!python -m pip install --upgrade bitsandbytes
!python -m pip uninstall -y torchao
!nvidia-smi

In [ ]:
import json
import os
import subprocess
from pathlib import Path

from google.colab import drive, userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/lad-generic-results')
OUTPUTS_DIR = DRIVE_ROOT / 'outputs'
RESULTS_DIR = DRIVE_ROOT / 'validation' / 'benchmark_runs'

# Compare hosted LLaDA, a mask-only adapter, and the legacy structured-noise model.
# Adapter paths are relative to OUTPUTS_DIR; hosted descriptors are used verbatim.
RUNS = [
    'llada:GSAI-ML/LLaDA-8B-Instruct',
    'llama-3.1-8b-mask-batch16/best',
    'legacy-hf:Ruurd/tini_model|diffusion-model-8B.pth',
]

# Full suite: set TASKS to ALL_TASKS below. For perplexity only, use ['open_ended'].
ALL_TASKS = ['mmlu', 'mmlu_pro', 'hellaswag', 'arc_c', 'gsm8k', 'math', 'gpqa', 'humaneval', 'mbpp', 'open_ended']
TASKS = ['open_ended']
LIMIT = None  # Optional absolute number of examples per task.
LIMIT_FRACTION = None  # Set to 0.05 for an evenly-spaced 1/20 smoke sample per task.
DEVICE = 'cuda'
QUANTIZATION = 'auto'  # 'auto', '4bit', or 'none'
LEGACY_TOKENIZER = 'meta-llama/Llama-3.2-3B'
INCLUDE_AUTOREGRESSIVE = False
# Independent 14B judge used only for standalone open-ended validation.
# This does not change training-time generation perplexity.
PERPLEXITY = {
    'model_name_or_path': 'microsoft/phi-4',
    'tokenizer_name_or_path': 'microsoft/phi-4',
    'quantization': '4bit',  # Change to 'none' only if the GPU has room for 14B BF16 weights.
    'precision': 'bf16',
    'cache_dir': '/content/base_models',
}

# Blind comparative quality judge. For N diffusion models, every prompt awards
# N-1 points to the best response down to 0 for the worst, with no ties.
# Candidate names are hidden and their order is shuffled before each API call.
OPEN_ENDED_JUDGE = {
    'enabled': True,
    'model': 'gpt-5',
    'methods': ['diffusion'],  # Add 'autoregressive' only if it should compete too.
    'reasoning_effort': 'medium',
    'seed': 1234,
    'fail_on_error': False,
}

# Settings shared by every inference family.
GENERATION_DEFAULTS = {'seed': 1234}

# The archived legacy model was trained with structured noise and keeps
# the corresponding iterative re-masking sampler. These settings do not
# affect hosted LLaDA or current mask-only adapters.
STRUCTURED_GENERATION = {
    'noise_level': 0.5,
    'temperature': 0.7,
    'top_k': 100,
    'proportional_unmask': True,
    'permanent_unmask': False,
    'confidence_guided': False,
}
STRUCTURED_TASK_GENERATION = {
    'open_ended': {'max_new_tokens': 256, 'num_steps': 256},
    'mmlu': {'max_new_tokens': 64, 'num_steps': 32},
    'mmlu_pro': {'max_new_tokens': 256, 'num_steps': 256},
    'hellaswag': {'max_new_tokens': 64, 'num_steps': 32},
    'arc_c': {'max_new_tokens': 64, 'num_steps': 32},
    'gsm8k': {'max_new_tokens': 512, 'num_steps': 512},
    'math': {'max_new_tokens': 512, 'num_steps': 512},
    'gpqa': {'max_new_tokens': 64, 'num_steps': 64},
    'humaneval': {'max_new_tokens': 512, 'num_steps': 512},
    'mbpp': {'max_new_tokens': 256, 'num_steps': 256},
}

# Hosted LLaDA and our mask-only adapter both use the official fixed-budget,
# low-confidence transfer sampler. Separate dictionaries allow intentional
# model-family overrides without changing the other family.
MASK_ONLY_GENERATION = {'temperature': 0.0, 'cfg_scale': 0.0, 'remasking': 'low_confidence'}
LLADA_GENERATION = {
    'temperature': 0.0, 'cfg_scale': 0.0, 'remasking': 'low_confidence',
    'eot_token_id': 126348,
}

# Published LLaDA task budgets. The short 3-token multiple-choice profiles
# are fast; MMLU-Pro and ARC-C deliberately use 256 and 512 forwards.
LLADA_STYLE_TASK_GENERATION = {
    'open_ended': {'max_new_tokens': 256, 'num_steps': 256, 'block_length': 256},
    'mmlu': {'max_new_tokens': 3, 'num_steps': 3, 'block_length': 3},
    'mmlu_pro': {'max_new_tokens': 256, 'num_steps': 256, 'block_length': 256},
    'hellaswag': {'max_new_tokens': 3, 'num_steps': 3, 'block_length': 3},
    'arc_c': {'max_new_tokens': 512, 'num_steps': 512, 'block_length': 512},
    'gsm8k': {'max_new_tokens': 512, 'num_steps': 512, 'block_length': 512, 'confidence_eos_eot_inf': True},
    'math': {'max_new_tokens': 512, 'num_steps': 512, 'block_length': 512, 'confidence_eos_eot_inf': True},
    'gpqa': {'max_new_tokens': 64, 'num_steps': 64, 'block_length': 64, 'confidence_eos_eot_inf': True},
    'humaneval': {'max_new_tokens': 512, 'num_steps': 512, 'block_length': 512, 'logits_eos_inf': True},
    'mbpp': {'max_new_tokens': 256, 'num_steps': 256, 'block_length': 256, 'confidence_eos_eot_inf': True},
}
MASK_ONLY_TASK_GENERATION = {task: dict(settings) for task, settings in LLADA_STYLE_TASK_GENERATION.items()}
LLADA_TASK_GENERATION = {task: dict(settings) for task, settings in LLADA_STYLE_TASK_GENERATION.items()}

print('Available adapters:')
from diffusion_lm.inference import find_adapters
print('\n'.join(find_adapters(OUTPUTS_DIR)))

In [ ]:
import yaml

config = {
    'outputs_dir': str(OUTPUTS_DIR),
    'models': RUNS,
    'tasks': TASKS,
    'split': 'test',
    'limit': LIMIT,
    'limit_fraction': LIMIT_FRACTION,
    'device': DEVICE,
    'quantization': QUANTIZATION,
    'legacy_tokenizer_name_or_path': LEGACY_TOKENIZER,
    'cache_dir': '/content/huggingface-datasets',
    'results_dir': str(RESULTS_DIR),
    'run_name': 'colab-validation',
    'include_autoregressive': INCLUDE_AUTOREGRESSIVE,
    'show_open_ended_answers': True,
    'perplexity': PERPLEXITY,
    'open_ended_judge': OPEN_ENDED_JUDGE,
    'generation': GENERATION_DEFAULTS,
    'generation_by_corruption': {
        'structured': STRUCTURED_GENERATION,
    },
    'task_generation_by_corruption': {
        'structured': STRUCTURED_TASK_GENERATION,
    },
    'mask_only_generation': MASK_ONLY_GENERATION,
    'mask_only_task_generation': MASK_ONLY_TASK_GENERATION,
    'llada_generation': LLADA_GENERATION,
    'llada_task_generation': LLADA_TASK_GENERATION,
}
config_path = Path('/content/benchmark_colab.yaml')
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(
    ['python', '-u', 'evaluate_benchmarks.py', '--config', str(config_path)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    bufsize=1, env=env,
)
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait() != 0:
    raise RuntimeError(f'Benchmark process failed with exit code {process.returncode}')

In [ ]:
import json
from pathlib import Path

import pandas as pd

run_dirs = sorted((path for path in RESULTS_DIR.iterdir() if path.is_dir()), reverse=True)
if not run_dirs:
    raise FileNotFoundError(f'No validation runs found under {RESULTS_DIR}')
latest_run = run_dirs[0]
summary = json.loads((latest_run / 'summary.json').read_text())
summary_rows = [result for model in summary['models'] for result in model['results']]
summary_frame = pd.DataFrame(summary_rows)
if not summary_frame.empty:
    summary_frame['display_method'] = summary_frame.apply(
        lambda row: 'autoregressive (original base)'
        if row.get('model_variant') == 'original_base' else row['method'],
        axis=1,
    )

# Accuracy exists for MMLU, ARC-C, code tasks, etc., but not for open_ended.
accuracy_rows = summary_frame.dropna(subset=['accuracy']) if 'accuracy' in summary_frame.columns else summary_frame.iloc[0:0]
if not accuracy_rows.empty:
    accuracy_table = (
        accuracy_rows.pivot_table(
            index='task', columns=['model', 'display_method'], values='accuracy', aggfunc='first'
        ).sort_index()
    )
    display(accuracy_table.style.format('{:.2%}'))
else:
    print('No accuracy results in this run (open_ended uses perplexity and repetition metrics).')

# Show open-ended quality metrics whenever that task was evaluated.
quality_rows = summary_frame.dropna(subset=['perplexity']) if 'perplexity' in summary_frame.columns else summary_frame.iloc[0:0]
if not quality_rows.empty:
    quality_columns = [
        column for column in (
            'model', 'method', 'evaluation_model', 'task', 'median_perplexity', 'mean_perplexity', 'perplexity', 'mean_nll', 'tokens',
            'mean_unigram_repetition', 'mean_bigram_repetition', 'mean_trigram_repetition',
        ) if column in quality_rows.columns
    ]
    quality_table = quality_rows[quality_columns].sort_values(['task', 'model', 'method'])
    display(quality_table.style.format({
        'median_perplexity': '{:.3f}', 'mean_perplexity': '{:.3f}', 'perplexity': '{:.3f}', 'mean_nll': '{:.3f}',
        'mean_unigram_repetition': '{:.2%}',
        'mean_bigram_repetition': '{:.2%}',
        'mean_trigram_repetition': '{:.2%}',
    }))

# GPT judge leaderboard (higher points are better).
judge_rows = summary_frame.dropna(subset=['judge_total_score']) if 'judge_total_score' in summary_frame.columns else summary_frame.iloc[0:0]
if not judge_rows.empty:
    judge_columns = [column for column in (
        'judge_leaderboard_position', 'model', 'method', 'judge_total_score',
        'judge_mean_score', 'judge_normalized_score', 'judge_first_place_count',
        'judge_comparisons', 'judge_model',
    ) if column in judge_rows.columns]
    judge_table = judge_rows[judge_columns].sort_values(
        ['judge_leaderboard_position', 'model', 'method']
    )
    display(judge_table.style.format({
        'judge_mean_score': '{:.3f}', 'judge_normalized_score': '{:.2%}',
    }))

result_files = sorted(latest_run.glob('models/**/results.jsonl'))
if result_files:
    records = [json.loads(line) for path in result_files for line in path.read_text().splitlines() if line.strip()]
    display(pd.DataFrame(records).head())
print(f'Report directory: {latest_run}')